In [6]:
import re
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [7]:
PASTA_BASE = Path(".")  # pasta Analysis (onde está o notebook)

re_cores = re.compile(r"Starting execution - cores=(\d+)")
re_seed  = re.compile(r"Running with seed=(\d+)")
re_gen   = re.compile(r"gen (\d+): ([\d.]+)s \| best=([^\s|]+) \| nfev=(\d+)")

dados = {}  # dados[seed][cores][gen] = {"time": ..., "best": ..., "nfev": ...}

for pasta in sorted(PASTA_BASE.glob("GRN5_*")):
    arquivo_log = pasta / "log.txt"
    if not arquivo_log.is_file():
        print(f"[aviso] log.txt não encontrado em {pasta.name}")
        continue

    cores = None
    seed = None
    with open(arquivo_log, encoding="utf-8") as f:
        for linha in f:
            m = re_cores.search(linha)
            if m:
                cores = int(m.group(1))
                continue
            m = re_seed.search(linha)
            if m:
                seed = int(m.group(1))
                continue
            m = re_gen.search(linha)
            if m and seed is not None and cores is not None:
                gen = int(m.group(1))
                dados.setdefault(seed, {}).setdefault(cores, {})[gen] = {
                    "time": float(m.group(2)),
                    "best": float(m.group(3)),
                    "nfev": int(m.group(4)),
                }

# Resumo da extração
for seed in sorted(dados):
    for cores in sorted(dados[seed]):
        gens = dados[seed][cores]
        print(f"seed={seed} | cores={cores:>2} | {len(gens)} gens | última gen={max(gens)}")

seed=0 | cores= 1 | 523 gens | última gen=20880
seed=0 | cores= 2 | 523 gens | última gen=20880
seed=0 | cores= 4 | 523 gens | última gen=20880
seed=0 | cores= 8 | 523 gens | última gen=20880
seed=0 | cores=16 | 523 gens | última gen=20880
seed=1 | cores= 1 | 1001 gens | última gen=40000
seed=1 | cores= 2 | 1001 gens | última gen=40000
seed=1 | cores= 4 | 1001 gens | última gen=40000
seed=1 | cores= 8 | 1001 gens | última gen=40000
seed=1 | cores=16 | 1001 gens | última gen=40000
seed=2 | cores= 1 | 1001 gens | última gen=40000
seed=2 | cores= 2 | 1001 gens | última gen=40000
seed=2 | cores= 4 | 1001 gens | última gen=40000
seed=2 | cores= 8 | 1001 gens | última gen=40000
seed=2 | cores=16 | 1001 gens | última gen=40000
seed=3 | cores= 1 | 1001 gens | última gen=40000
seed=3 | cores= 2 | 1001 gens | última gen=40000
seed=3 | cores= 4 | 1001 gens | última gen=40000
seed=3 | cores= 8 | 1001 gens | última gen=40000
seed=3 | cores=16 | 1001 gens | última gen=40000


In [8]:
METRICAS = ["time", "best", "nfev"]

def montar_arrays(dados, seeds):
    """Monta arrays (n_cores, n_gens) para cada métrica, com média entre as seeds dadas.
    Posições sem dado ficam NaN."""
    cores = sorted({c for s in seeds for c in dados[s]})
    gens  = sorted({g for s in seeds for c in dados[s] for g in dados[s][c]})
    i_core = {c: i for i, c in enumerate(cores)}
    i_gen  = {g: j for j, g in enumerate(gens)}

    soma = {m: np.zeros((len(cores), len(gens))) for m in METRICAS}
    cont = np.zeros((len(cores), len(gens)))

    for s in seeds:
        for c, por_gen in dados[s].items():
            for g, v in por_gen.items():
                i, j = i_core[c], i_gen[g]
                for m in METRICAS:
                    soma[m][i, j] += v[m]
                cont[i, j] += 1

    with np.errstate(invalid="ignore", divide="ignore"):
        arrays = {m: np.where(cont > 0, soma[m] / cont, np.nan) for m in METRICAS}

    return np.array(cores), np.array(gens), arrays

# Seed 0 (separada)
cores_s0, gens_s0, arr_s0 = montar_arrays(dados, [0])
time_s0, best_s0, nfev_s0 = arr_s0["time"], arr_s0["best"], arr_s0["nfev"]

# Demais seeds (média)
seeds_resto = sorted(s for s in dados if s != 0)
cores_med, gens_med, arr_med = montar_arrays(dados, seeds_resto)
time_med, best_med, nfev_med = arr_med["time"], arr_med["best"], arr_med["nfev"]

print(f"Seed 0        -> cores={cores_s0.tolist()} | gens: {len(gens_s0)} (até {gens_s0[-1]}) | shape {time_s0.shape}")
print(f"Seeds {seeds_resto} (média) -> cores={cores_med.tolist()} | gens: {len(gens_med)} (até {gens_med[-1]}) | shape {time_med.shape}")
print(f"NaN na seed 0: {np.isnan(time_s0).sum()} | NaN na média: {np.isnan(time_med).sum()}")

Seed 0        -> cores=[1, 2, 4, 8, 16] | gens: 523 (até 20880) | shape (5, 523)
Seeds [1, 2, 3] (média) -> cores=[1, 2, 4, 8, 16] | gens: 1001 (até 40000) | shape (5, 1001)
NaN na seed 0: 0 | NaN na média: 0


In [9]:
# ===================== CONFIGURAÇÕES DO PLOT (edite à vontade) =====================
COLORSCALE    = "Plasma"   # escala de cores da superfície: "Plasma", "Cividis", "Turbo", "Blues", "Viridis"...

WIREFRAME     = True       # liga/desliga o wireframe
WIRE_COLOR    = "black"    # cor das linhas do wireframe
WIRE_WIDTH    = 1          # espessura das linhas (1 a 16)
WIRE_STEP_GEN = 1000       # espaçamento das linhas no eixo das gerações

PONTOS        = True       # liga/desliga as bolinhas nos pontos reais
PONTO_COLOR   = "red"      # cor das bolinhas
PONTO_SIZE    = 2          # tamanho das bolinhas (px)
PONTO_STEP    = 1          # 1 = todos os pontos; N = um a cada N gerações registradas
# ===================================================================================


def _tracos(cores, gens, Z, z_label, x_max=None, coloraxis=None):
    """Cria os traços (superfície + bolinhas) de um gráfico."""
    cores = np.asarray(cores)
    gens  = np.asarray(gens)
    x_max = gens.max() if x_max is None else x_max
    y_idx = np.arange(len(cores))  # cores igualmente espaçados (1, 2, 4, 8, 16)

    hover = (
        "Geração: %{x}<br>"
        "Cores: %{customdata}<br>"
        f"{z_label}: " + "%{z}<extra></extra>"
    )

    # Wireframe (linhas da grade desenhadas sobre a superfície colorida)
    contornos = {}
    if WIREFRAME:
        contornos = dict(
            x=dict(show=True, start=0, end=x_max, size=WIRE_STEP_GEN,
                   color=WIRE_COLOR, width=WIRE_WIDTH),
            y=dict(show=True, start=0, end=len(cores) - 1, size=1,
                   color=WIRE_COLOR, width=WIRE_WIDTH),
        )

    cor = dict(coloraxis=coloraxis) if coloraxis else dict(colorscale=COLORSCALE, colorbar=dict(title=z_label))
    tracos = [go.Surface(
        x=gens,
        y=y_idx,
        z=Z,
        customdata=np.tile(cores[:, None], (1, len(gens))),
        hovertemplate=hover,
        contours=contornos,
        name="superfície",
        **cor,
    )]

    # Bolinhas nos pontos reais
    if PONTOS:
        cols = np.arange(0, len(gens), PONTO_STEP)
        X = np.tile(gens[cols], len(cores))
        Y = np.repeat(y_idx, len(cols))
        C = np.repeat(cores, len(cols))
        Zp = Z[:, cols].ravel()
        ok = ~np.isnan(Zp)
        tracos.append(go.Scatter3d(
            x=X[ok], y=Y[ok], z=Zp[ok],
            mode="markers",
            marker=dict(color=PONTO_COLOR, size=PONTO_SIZE),
            customdata=C[ok],
            hovertemplate=hover,
            name="pontos reais",
            showlegend=False,
        ))
    return tracos


def _cena(cores, z_label, x_range=None, z_range=None):
    """Configuração dos eixos de uma cena 3D."""
    y_idx = np.arange(len(cores))
    return dict(
        xaxis=dict(title="Geração", range=x_range),
        yaxis=dict(title="Nº de cores", tickvals=y_idx, ticktext=[str(c) for c in cores]),
        zaxis=dict(title=z_label, range=z_range),
        aspectmode="manual",
        aspectratio=dict(x=2.5, y=1, z=1),  # base retangular
    )


# ----------------------------- Gráfico individual -----------------------------
def plot_superficie(cores, gens, Z, titulo, z_label):
    """Superfície 3D: base (gerações x cores), altura = Z com shape (n_cores, n_gens)."""
    fig = go.Figure(_tracos(cores, gens, Z, z_label))
    fig.update_layout(title=titulo, width=1000, height=700, scene=_cena(cores, z_label))
    fig.show()
    return fig

def plot_tempo(cores, gens, time_arr, rotulo):
    return plot_superficie(cores, gens, time_arr, f"Tempo decorrido — {rotulo}", "Tempo (s)")

def plot_nfev(cores, gens, nfev_arr, rotulo):
    return plot_superficie(cores, gens, nfev_arr, f"Nº de avaliações (nfev) — {rotulo}", "nfev")

def plot_best(cores, gens, best_arr, rotulo):
    return plot_superficie(cores, gens, best_arr, f"Melhor fitness (best) — {rotulo}", "Best (MSE)")


# ----------------------------- Lado a lado -----------------------------
def plot_lado_a_lado(cores_a, gens_a, Z_a, rotulo_a,
                     cores_b, gens_b, Z_b, rotulo_b,
                     titulo, z_label):
    """Dois gráficos 3D lado a lado, com mesma escala de gerações, de Z e de cores."""
    x_max = max(np.max(gens_a), np.max(gens_b))
    z_min = min(np.nanmin(Z_a), np.nanmin(Z_b))
    z_max = max(np.nanmax(Z_a), np.nanmax(Z_b))
    folga = 0.02 * (z_max - z_min) if z_max > z_min else 1e-9
    z_range = [z_min - folga, z_max + folga]

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(rotulo_a, rotulo_b),
        horizontal_spacing=0.02,
    )
    for tr in _tracos(cores_a, gens_a, Z_a, z_label, x_max=x_max, coloraxis="coloraxis"):
        fig.add_trace(tr, row=1, col=1)
    for tr in _tracos(cores_b, gens_b, Z_b, z_label, x_max=x_max, coloraxis="coloraxis"):
        fig.add_trace(tr, row=1, col=2)

    fig.update_layout(
        title=titulo,
        width=1700,
        height=650,
        coloraxis=dict(colorscale=COLORSCALE, cmin=z_min, cmax=z_max, colorbar=dict(title=z_label)),
        scene=_cena(cores_a, z_label, x_range=[0, x_max], z_range=z_range),
        scene2=_cena(cores_b, z_label, x_range=[0, x_max], z_range=z_range),
    )
    fig.show()
    return fig

In [10]:
plot_lado_a_lado(cores_s0, gens_s0, time_s0, "Seed 0",
                 cores_med, gens_med, time_med, "Média seeds 1–3",
                 "Tempo decorrido — Seed 0 vs média", "Tempo (s)")

plot_lado_a_lado(cores_s0, gens_s0, nfev_s0, "Seed 0",
                 cores_med, gens_med, nfev_med, "Média seeds 1–3",
                 "Nº de avaliações (nfev) — Seed 0 vs média", "nfev")

plot_lado_a_lado(cores_s0, gens_s0, best_s0, "Seed 0",
                 cores_med, gens_med, best_med, "Média seeds 1–3",
                 "Melhor fitness (best) — Seed 0 vs média", "Best (MSE)")